In [2]:
# -*- coding: utf-8 -*-
"""Multimodal Image Analyzer (Gradio Web UI in Colab)

在 Colab 里运行后会生成一个网页链接，可以直接拖拽/上传图片，
AI 会用 Llama-3.2 Vision 模型分析图片内容。
"""

# 1. 安装依赖
!pip install gradio requests

import gradio as gr
import requests
import base64
from io import BytesIO
from google.colab import userdata

# 2. 配置 API Key
OPENROUTER_API_KEY = userdata.get('OPENROUTER_API_KEY')

# 3. 图片转 Base64
def encode_image(pil_image):
    buffered = BytesIO()
    # 统一转成 RGB 再存成 JPEG，避免 PNG/RGBA 等格式报错
    pil_image = pil_image.convert("RGB")
    pil_image.save(buffered, format="JPEG")
    return base64.b64encode(buffered.getvalue()).decode("utf-8")

# 4. 调用视觉大模型分析图片
def analyze_image(image, user_prompt):
    if image is None:
        return "请先上传一张图片。"

    image_base64 = encode_image(image)

    prompt_text = user_prompt.strip() if user_prompt.strip() else \
        "请详细分析这张图片。如果是体育赛事，请告诉我里面有什么车队/球队、运动员在做什么动作、以及画面中的核心焦点是什么。"

    try:
        response = requests.post(
            url="https://openrouter.ai/api/v1/chat/completions",
            headers={
                "Authorization": f"Bearer {OPENROUTER_API_KEY}",
                "Content-Type": "application/json"
            },
            json={
                "model": "meta-llama/llama-3.2-11b-vision-instruct",
                "messages": [
                    {
                        "role": "user",
                        "content": [
                            {"type": "text", "text": prompt_text},
                            {
                                "type": "image_url",
                                "image_url": {"url": f"data:image/jpeg;base64,{image_base64}"}
                            }
                        ]
                    }
                ]
            },
            timeout=60
        )

        result = response.json()

        if "choices" not in result:
            return f"[出错] API 返回异常：\n{result}"

        return result["choices"][0]["message"]["content"]

    except Exception as e:
        return f"[出错] 请求失败：{e}"

# 5. 构建 Gradio 网页界面
with gr.Blocks(title="多模态图片分析器") as demo:
    gr.Markdown("# 🖼️ 多模态视觉理解 Demo (Llama-3.2 Vision)")
    gr.Markdown("上传一张图片（比如 F1 赛车、足球比赛画面），AI 会自动分析画面内容。")

    with gr.Row():
        with gr.Column():
            image_input = gr.Image(type="pil", label="上传图片")
            prompt_input = gr.Textbox(
                label="自定义提问（留空则使用默认分析提示）",
                placeholder="例如：这张图里的车队是谁？车手在做什么动作？",
                lines=2
            )
            submit_btn = gr.Button("开始分析", variant="primary")
        with gr.Column():
            output_box = gr.Textbox(label="AI 分析结果", lines=15)

    submit_btn.click(fn=analyze_image, inputs=[image_input, prompt_input], outputs=output_box)

# 6. 启动网页，share=True 会生成一个公开可访问的链接
demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://be6fd2e5a4401970a0.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


ERROR:    Exception in ASGI application
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/protocols/http/httptools_impl.py", line 421, in run_asgi
    result = await app(  # type: ignore[func-returns-value]
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/middleware/proxy_headers.py", line 62, in __call__
    return await self.app(scope, receive, send)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/fastapi/applications.py", line 1159, in __call__
    await super().__call__(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/applications.py", line 107, in __call__
    await self.middleware_stack(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/errors.py", line 186, in __call__
    raise exc
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/error

Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://be6fd2e5a4401970a0.gradio.live
